<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>GraphRAG</h1>
<h1>From Text to Knowledge Graphs with REBEL, fastcoref &amp; Neo4j</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In this notebook we build an end-to-end pipeline that turns raw Wikipedia text into a queryable **Knowledge Graph**:

1. **Download** the `wikitext-103-v1` dataset from Hugging Face.
2. **Explore** the data with basic statistics and visualizations.
3. **Resolve coreference** with [`fastcoref`](https://github.com/shon-otmazgin/fastcoref) so that pronouns like *"He"* are replaced by the entity they refer to.
4. **Extract triplets** `(head, relation, tail)` with the [`Babelscape/rebel-large`](https://huggingface.co/Babelscape/rebel-large) sequence-to-sequence model.
5. **Link** every entity to its [Wikidata](https://www.wikidata.org) QID through the public MediaWiki API.
6. **Ingest** the resulting graph into **Neo4j** (with an automatic NetworkX fallback for environments where Neo4j is not available).
7. **Query** the knowledge graph with Cypher and visualize it interactively with `pyvis`.

Heavy steps (model inference, API calls) are cached to disk under `./cache/` so the notebook can be re-run quickly.

In [1]:
import json
import os
import pickle
import re
import time
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

import anthropic
from anthropic import Anthropic

import pyvis
from pyvis.network import Network
from IPython.display import IFrame

import transformers
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import fastcoref
from fastcoref import FCoref

import datasets
from datasets import load_dataset

import pyvis
import spacy
import torch
import transformers

import watermark

%load_ext watermark
%matplotlib inline

We start by printing out the versions of the libraries we're using for future reference.

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.12.13
IPython version      : 9.15.0

Compiler    : Clang 22.1.3 
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 0c49bafe5b4c495e03a32e43559f61efb9f6ab2f

IPython     : 9.15.0
anthropic   : 0.116.0
datasets    : 5.0.0
fastcoref   : 2.1.6
json        : 2.0.9
matplotlib  : 3.11.0
networkx    : 3.6.1
numpy       : 2.5.1
pandas      : 3.0.3
pyvis       : 0.3.2
re          : 2.2.1
requests    : 2.34.2
spacy       : 3.8.14
torch       : 2.12.1
tqdm        : 4.68.3
transformers: 5.13.0
watermark   : 2.6.0



Load the default Data For Science figure style.

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

## Configuration

All knobs of the pipeline are grouped here so they are easy to tweak. The defaults are chosen so the notebook finishes in a few minutes on a laptop (Apple Silicon MPS / CPU).

In [4]:
CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)

# Use None to process every article (~28k - slow!) or set an int for a quick demo.
NUM_ARTICLES = None
REBEL_BATCH_SIZE = 4
REBEL_MAX_INPUT_TOKENS = 256
REBEL_MAX_OUTPUT_TOKENS = 256
REBEL_NUM_BEAMS = 3

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

USER_AGENT = 'KGTutorial/0.1 (info@data4sci.com)'

print(f'Using device: {DEVICE}')
print(f'Cache directory: {CACHE_DIR.resolve()}')

Using device: mps
Cache directory: /Users/bgoncalves/My Drive/DataForScience/GraphRAG/cache
Neo4j target: neo4j://127.0.0.1:7687 (user='neo4j')


# 1. Download WikiText-103-v1

[`wikitext-103-v1`](https://huggingface.co/datasets/wikitext) is a collection of more than 100 million tokens extracted from the *Good* and *Featured* articles of English Wikipedia. We grab the training split through the Hugging Face `datasets` library and group lines back into articles.

Each line in the dataset is either:
* an article title: `= Title =`
* a section header: `= = Section = =` (or more `=`s for sub-sections)
* a paragraph of running text
* the empty string (a paragraph separator)

We also need to undo WikiText's mild tokenization (`@-@`, `@,@`, `@.@`, `<unk>`).

In [5]:
ARTICLE_HEADER_RE = re.compile(r'^\s*=\s+([^=].*?)\s+=\s*$')
SECTION_HEADER_RE = re.compile(r'^\s*=+\s+.+?\s+=+\s*$')

def is_article_header(line: str) -> bool:
    """True when the line is a top-level WikiText article header (single = on each side)."""
    return ARTICLE_HEADER_RE.match(line) is not None

def clean_wikitext(text: str) -> str:
    """Undo WikiText's special tokenization."""
    text = text.replace(' @-@ ', '-').replace(' @,@ ', ',').replace(' @.@ ', '.')
    text = text.replace('<unk>', '')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def stream_articles(num_articles: int | None = None):
    """Yield parsed articles from wikitext-103-v1 train.

    If *num_articles* is None, every article in the split is yielded.
    Each article keeps the full text of every paragraph.
    """
    ds = load_dataset('wikitext', 'wikitext-103-v1', split='train', streaming=True)
    current = None
    yielded = 0
    for row in ds:
        line = row['text']
        if is_article_header(line):
            if current is not None and current['paragraphs']:
                yield current
                yielded += 1
                if num_articles is not None and yielded >= num_articles:
                    return
            current = {'title': ARTICLE_HEADER_RE.match(line).group(1), 'paragraphs': []}
        elif SECTION_HEADER_RE.match(line):
            continue  # skip section/sub-section headers
        else:
            text = clean_wikitext(line)
            if text and current is not None:
                current['paragraphs'].append(text)
    if current is not None and current['paragraphs']:
        yield current

In [6]:
articles_cache = CACHE_DIR / f"articles_{NUM_ARTICLES if NUM_ARTICLES is not None else 'all'}.pkl"

if articles_cache.exists():
    with open(articles_cache, 'rb') as f:
        articles = pickle.load(f)
    print(f'Loaded {len(articles)} articles from cache.')
else:
    articles = list(tqdm(stream_articles(NUM_ARTICLES), total=NUM_ARTICLES, desc='Streaming WikiText'))
    with open(articles_cache, 'wb') as f:
        pickle.dump(articles, f)
    print(f'Cached {len(articles)} articles to {articles_cache}.')

print('First article:', articles[0]['title'])
print('Paragraphs   :', len(articles[0]['paragraphs']))
print('Snippet      :', articles[0]['paragraphs'][0][:300], '...')

Streaming WikiText: 0it [00:00, ?it/s]

07/05/2026 16:42:41 - INFO - 	 HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
07/05/2026 16:42:41 - INFO - 	 HTTP Request: HEAD https://huggingface.co/datasets/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
07/05/2026 16:42:41 - INFO - 	 HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
07/05/2026 16:42:41 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Salesforce/wikitext/b08601e04326c79dfdd32d625aee71d232d685c3/README.md "HTTP/1.1 200 OK"
07/05/2026 16:42:41 - INFO - 	 HTTP Request: HEAD https://huggingface.co/datasets/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 307 Temporary Redirect"
07/05/2026 16:42:41 - INFO - 	 HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 404 Not Found"
07/05/2026 16:42:41 

ValueError: Couldn't find cache for wikitext for config 'wikitext-103-v1'
Available configs in the cache: ['wikitext-103-raw-v1']

In [ ]:
[i for i, art in enumerate(articles) if "Newton" in art["title"]]

In [ ]:
articles[721]['title']

# 2. Exploratory Data Analysis

Before we throw the heavy NLP machinery at the corpus, let's get a feel for what we're dealing with.

In [ ]:
stats_rows = []
for art in articles:
    text = ' '.join(art['paragraphs'])
    stats_rows.append({
        'title': art['title'],
        'n_paragraphs': len(art['paragraphs']),
        'n_chars': len(text),
        'n_words': len(text.split()),
    })

stats_df = pd.DataFrame(stats_rows)
print(f'Articles : {len(stats_df)}')
print(f'Total words : {stats_df.n_words.sum():,}')
print(f'Total chars : {stats_df.n_chars.sum():,}')
stats_df.describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(stats_df['n_words'], bins=15, color=colors[0], edgecolor='white')
axes[0].set_title('Words per article')
axes[0].set_xlabel('# words')
axes[0].set_ylabel('# articles')

axes[1].hist(stats_df['n_paragraphs'], bins=30,
             color=colors[1], edgecolor='white')
axes[1].set_title('Paragraphs per article')
axes[1].set_xlabel('# paragraphs')
axes[1].set_ylabel('# articles')

plt.show()

In [ ]:
STOPWORDS = set('''the a an of and in to with for on at by from is are was were be been being
    it its as that this these those he she they we i his her their our its but or not
    s have has had which who whom whose what when where why how than then so if also into
    over under between among through about against during before after above below up down
    out off only own same other any all each more most some such no nor too very just one
    two three first second third'''.split())

word_counts = Counter()
for art in articles:
    for para in art['paragraphs']:
        for token in re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", para.lower()):
            if token not in STOPWORDS and len(token) > 2:
                word_counts[token] += 1

top_words = pd.DataFrame(word_counts.most_common(20), columns=['word', 'count'])

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_words['word'][::-1], top_words['count'][::-1], color=colors[2])
ax.set_title('Top 20 content words')
ax.set_xlabel('frequency')
plt.show()

# 3. Coreference Resolution with fastcoref

Plain text is full of pronouns. *"Obama was born in Hawaii. **He** later became president."* A relation extractor that processes the second sentence in isolation has no way of knowing that *He* refers to **Obama**.

[`fastcoref`](https://github.com/shon-otmazgin/fastcoref) is an efficient end-to-end coreference resolver. We use it to **rewrite** each article so that every mention is replaced by the longest non-pronoun mention in its cluster, which gives REBEL the best chance of producing high-quality triplets.

In [ ]:
fcoref_model = FCoref(device=COREF_DEVICE, enable_progress_bar=False)

In [ ]:
PRONOUNS = {
    'he', 'she', 'it', 'they', 'him', 'her', 'them', 'his', 'hers', 'its', 'their',
    'theirs', 'i', 'me', 'my', 'mine', 'we', 'us', 'our', 'ours', 'you', 'your',
    'yours', 'this', 'that', 'these', 'those', 'himself', 'herself', 'itself',
    'themselves', 'who', 'whom', 'whose', 'which',
}

def pick_representative(mentions):
    """Pick the longest non-pronoun mention as cluster representative."""
    non_pron = [m for m in mentions if m.lower().strip() not in PRONOUNS]
    if not non_pron:
        return mentions[0]
    return max(non_pron, key=len)

def resolve_with_coref(text: str, pred) -> str:
    """Replace every mention in *text* with the cluster's representative."""
    # fastcoref's get_clusters(as_strings=False) can include None for mentions that
    # did not align to character offsets; get_clusters(as_strings=True) omits those.
    # Build (span, surface) pairs from pred.clusters so strings and spans stay aligned.
    edits = []
    for cluster in pred.clusters:
        pairs = []
        for mention in cluster:
            _idx, ch_span = pred.char_map[mention]
            if ch_span is None:
                continue
            start, end = ch_span
            pairs.append(((start, end), text[start:end]))
        if not pairs:
            continue
        rep = pick_representative([s for _, s in pairs])
        for (start, end), _ in pairs:
            edits.append((start, end, rep))
    edits.sort(key=lambda x: x[0], reverse=True)
    chars = list(text)
    for start, end, rep in edits:
        chars[start:end] = rep
    return ''.join(chars)

In [ ]:
coref_cache = CACHE_DIR / f"coref_{NUM_ARTICLES if NUM_ARTICLES is not None else 'all'}.pkl"

if coref_cache.exists():
    with open(coref_cache, 'rb') as f:
        resolved_articles = pickle.load(f)
    print(f'Loaded {len(resolved_articles)} resolved articles from cache.')
else:
    article_texts = [' '.join(a['paragraphs']) for a in articles]
    preds = fcoref_model.predict(texts=article_texts)
    resolved_articles = []
    for art, raw, pred in zip(articles, article_texts, preds):
        resolved_articles.append({
            'title': art['title'],
            'raw': raw,
            'resolved': resolve_with_coref(raw, pred),
            'clusters': pred.get_clusters(as_strings=True),
        })
    with open(coref_cache, 'wb') as f:
        pickle.dump(resolved_articles, f)
    print(f'Cached {len(resolved_articles)} resolved articles to {coref_cache}.')

In [ ]:
sample = resolved_articles[0]
print('Title         :', sample['title'])
print('# clusters   :', len(sample['clusters']))
print('First clusters:')
pprint(sample['clusters'][:5])
print('\n--- BEFORE (chars 0-450) ---\n')
print(sample['raw'][:450])
print('\n--- AFTER  (chars 0-450) ---\n')
print(sample['resolved'][:450])

# 4. Relation Extraction with REBEL

[REBEL](https://github.com/Babelscape/rebel) (Huguet Cabot & Navigli, 2021) is a BART-based seq2seq model that reads a piece of text and emits a linearized stream of triplets:

```
<triplet> SUBJECT <subj> OBJECT <obj> RELATION  <triplet> ...
```

We chunk each resolved article into sentence-windowed paragraphs of at most `REBEL_MAX_INPUT_TOKENS` tokens, run the model in batches, and decode the output into Python dictionaries.

In [ ]:
REBEL_MODEL = 'Babelscape/rebel-large'
rebel_tokenizer = AutoTokenizer.from_pretrained(REBEL_MODEL)
rebel_model = AutoModelForSeq2SeqLM.from_pretrained(REBEL_MODEL).to(DEVICE)
rebel_model.eval()
print('Loaded', REBEL_MODEL, 'on', DEVICE)

In [ ]:
def parse_rebel_output(decoded: str):
    """Parse a REBEL decoded string into a list of {head, type, tail} dicts."""
    triplets = []
    relation, subject, object_, current = '', '', '', None
    text = decoded.replace('<s>', '').replace('<pad>', '').replace('</s>', '').strip()
    for token in text.split():
        if token == '<triplet>':
            current = 't'
            if relation:
                triplets.append({'head': subject.strip(), 'type': relation.strip(), 'tail': object_.strip()})
                relation = ''
            subject = ''
        elif token == '<subj>':
            current = 's'
            if relation:
                triplets.append({'head': subject.strip(), 'type': relation.strip(), 'tail': object_.strip()})
            object_ = ''
        elif token == '<obj>':
            current = 'o'
            relation = ''
        else:
            if current == 't':
                subject += ' ' + token
            elif current == 's':
                object_ += ' ' + token
            elif current == 'o':
                relation += ' ' + token
    if subject and relation and object_:
        triplets.append({'head': subject.strip(), 'type': relation.strip(), 'tail': object_.strip()})
    return triplets

def split_into_chunks(text: str, max_tokens: int = REBEL_MAX_INPUT_TOKENS) -> list[str]:
    """Greedy sentence-level chunker bounded by REBEL's tokenizer length."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, buf, buf_tokens = [], [], 0
    for sent in sentences:
        if not sent:
            continue
        n_tokens = len(rebel_tokenizer.tokenize(sent))
        if buf_tokens + n_tokens > max_tokens and buf:
            chunks.append(' '.join(buf))
            buf, buf_tokens = [], 0
        buf.append(sent)
        buf_tokens += n_tokens
    if buf:
        chunks.append(' '.join(buf))
    return chunks

@torch.no_grad()
def rebel_extract(chunks: list[str], batch_size: int = REBEL_BATCH_SIZE) -> list[list[dict]]:
    """Run REBEL on a list of text chunks and return their triplets."""
    all_triplets = []
    for i in tqdm(range(0, len(chunks), batch_size), desc='REBEL'):
        batch = chunks[i:i + batch_size]
        inputs = rebel_tokenizer(batch, return_tensors='pt', padding=True,
                                 truncation=True, max_length=REBEL_MAX_INPUT_TOKENS).to(DEVICE)
        generated = rebel_model.generate(
            **inputs,
            max_length=REBEL_MAX_OUTPUT_TOKENS,
            num_beams=REBEL_NUM_BEAMS,
            num_return_sequences=1,
            length_penalty=1.0,
        )
        decoded = rebel_tokenizer.batch_decode(generated, skip_special_tokens=False)
        for d in decoded:
            all_triplets.append(parse_rebel_output(d))
    return all_triplets

In [ ]:
triplets_cache = CACHE_DIR / f"triplets_{NUM_ARTICLES if NUM_ARTICLES is not None else 'all'}.parquet"

if triplets_cache.exists():
    triplets_df = pd.read_parquet(triplets_cache)
    print(f'Loaded {len(triplets_df):,} cached triplets.')
else:
    rows = []
    for art in tqdm(resolved_articles, desc='Articles'):
        chunks = split_into_chunks(art['resolved'])
        triplets_per_chunk = rebel_extract(chunks)
        for chunk_idx, (chunk, trips) in enumerate(zip(chunks, triplets_per_chunk)):
            for trip in trips:
                rows.append({
                    'article': art['title'],
                    'chunk_idx': chunk_idx,
                    'chunk_text': chunk,
                    **trip,
                })
    triplets_df = pd.DataFrame(rows)
    triplets_df.to_parquet(triplets_cache, index=False)
    print(f'Extracted and cached {len(triplets_df):,} triplets to {triplets_cache}.')

In [ ]:
print(f'Total triplets       : {len(triplets_df):,}')
print(f'Unique relations    : {triplets_df["type"].nunique()}')
print(f'Unique entities (h+t): {pd.unique(triplets_df[["head", "tail"]].values.ravel()).shape[0]}')
print()
print('Top 15 relations:')
print(triplets_df['type'].value_counts().head(15).to_string())
triplets_df.head(10)

In [ ]:
top_rel = triplets_df['type'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_rel.index[::-1], top_rel.values[::-1], color=colors[3])
ax.set_title('Most frequent REBEL relations')
ax.set_xlabel('count')
plt.show()

# 5. Entity Linking to Wikidata

REBEL returns raw entity strings such as *"Barack Obama"* or *"Hawaii"*. To make the graph **interoperable** with the rest of the linked-open-data world we map each entity to its [Wikidata](https://www.wikidata.org) QID via the public `wbsearchentities` MediaWiki API.

We cache every API call so that re-runs are free and we stay polite with the Wikidata servers.

In [ ]:
wikidata_cache_file = CACHE_DIR / 'wikidata_lookup.json'
if wikidata_cache_file.exists():
    with open(wikidata_cache_file) as f:
        wikidata_cache = json.load(f)
else:
    wikidata_cache = {}

def wikidata_lookup(entity: str, session: requests.Session | None = None) -> dict | None:
    """Return {qid, label, description, url} for the top match (or None)."""
    if entity in wikidata_cache:
        return wikidata_cache[entity]
    sess = session or requests
    try:
        resp = sess.get(
            'https://www.wikidata.org/w/api.php',
            params={
                'action': 'wbsearchentities',
                'search': entity,
                'language': 'en',
                'format': 'json',
                'limit': 1,
            },
            headers={'User-Agent': USER_AGENT},
            timeout=10,
        )
        data = resp.json()
        if data.get('search'):
            s = data['search'][0]
            result = {
                'qid': s['id'],
                'label': s.get('label', entity),
                'description': s.get('description', ''),
                'url': s.get('concepturi', ''),
            }
        else:
            result = None
    except Exception as exc:
        print(f'lookup failed for {entity!r}: {exc}')
        result = None
    wikidata_cache[entity] = result
    return result

def save_wikidata_cache():
    with open(wikidata_cache_file, 'w') as f:
        json.dump(wikidata_cache, f, indent=2)

In [ ]:
unique_entities = sorted(set(triplets_df['head']).union(triplets_df['tail']))
print(f'Linking {len(unique_entities):,} unique entities to Wikidata...')

session = requests.Session()
for ent in tqdm(unique_entities, desc='Wikidata'):
    wikidata_lookup(ent, session=session)
    time.sleep(0.05)  # be polite
save_wikidata_cache()

linked = sum(1 for v in wikidata_cache.values() if v)
print(f'Linked {linked}/{len(unique_entities)} entities ({linked/len(unique_entities):.0%}).')

In [ ]:
linked_examples = [(e, wikidata_cache[e]) for e in unique_entities if wikidata_cache.get(e)][:15]
linked_df = pd.DataFrame([
    {'entity': e, 'qid': v['qid'], 'label': v['label'], 'description': v['description']}
    for e, v in linked_examples
])
linked_df

# 6. Build the Knowledge Graph in Neo4j

We now have everything we need to push the data into a graph database. The cell below connects to **Neo4j at `neo4j://127.0.0.1:7687`** by default (the same host/port Neo4j Browser uses for a local single instance). Override with the `NEO4J_URI` environment variable if your install differs.

If the connection fails (server down or wrong password) we fall back to an in-memory NetworkX graph so the rest of the notebook still runs.

### Starting a local Neo4j

Pick whichever is easiest for you:

```bash
# Option A - Docker (one command, throw-away)
docker run --rm -d --name nodes2026-neo4j \
    -p 7474:7474 -p 7687:7687 \
    -e NEO4J_AUTH=neo4j/nodes2026 \
    -e NEO4J_PLUGINS='["apoc"]' \
    neo4j:5

# Option B - Neo4j Desktop or AuraDB Free
#   then set NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD env vars before launching Jupyter.
```

The default password in this notebook is `nodes2026`, matching the Docker command above. If you use Neo4j Desktop’s initial password or another install, set `NEO4J_PASSWORD` (and `NEO4J_USER` if not `neo4j`) so they match your database.

In [ ]:
neo4j_driver = None
USE_NEO4J = False
try:
    # neo4j:// is the recommended scheme for Neo4j 4+ (routing-aware; works for a single local instance too).
    neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    neo4j_driver.verify_connectivity()
    USE_NEO4J = True
    print(f'Connected to Neo4j at {NEO4J_URI} as {NEO4J_USER!r}')
except AuthError as exc:
    print(f'Neo4j authentication failed for {NEO4J_URI!r} / user {NEO4J_USER!r}.')
    print('The server is reachable but the password is wrong. Set NEO4J_PASSWORD in the')
    print('environment (or change the default in the Configuration cell) to match your DB.')
    print(f'Details: {exc}')
    if neo4j_driver is not None:
        neo4j_driver.close()
        neo4j_driver = None
    print('Falling back to an in-memory NetworkX graph.')
except ServiceUnavailable as exc:
    print(f'Neo4j not reachable at {NEO4J_URI!r} ({exc}).')
    print('Start Neo4j locally (Docker, Desktop, or brew) and confirm it listens on 7687.')
    if neo4j_driver is not None:
        neo4j_driver.close()
        neo4j_driver = None
    print('Falling back to an in-memory NetworkX graph.')
except Exception as exc:
    print(f'Neo4j connection error ({exc.__class__.__name__}: {exc}).')
    print('Falling back to an in-memory NetworkX graph - the same Cypher demos are')
    print('translated into NetworkX below so the notebook still runs end-to-end.')
    if neo4j_driver is not None:
        neo4j_driver.close()
        neo4j_driver = None

print('USE_NEO4J =', USE_NEO4J)

In [ ]:
def relation_to_label(rel: str) -> str:
    """Normalize a free-form REBEL relation into a Cypher-safe REL_TYPE."""
    rel = re.sub(r'[^A-Za-z0-9]+', '_', rel.strip())
    rel = re.sub(r'_+', '_', rel).strip('_').upper()
    return rel or 'RELATED_TO'

kg_nx = nx.MultiDiGraph()

def add_entity_to_nx(name: str):
    if kg_nx.has_node(name):
        return
    info = wikidata_cache.get(name) or {}
    kg_nx.add_node(
        name,
        qid=info.get('qid'),
        label=info.get('label', name),
        description=info.get('description', ''),
        url=info.get('url', ''),
    )

for _, row in triplets_df.iterrows():
    add_entity_to_nx(row['head'])
    add_entity_to_nx(row['tail'])
    kg_nx.add_edge(row['head'], row['tail'],
                   relation=row['type'], rel_label=relation_to_label(row['type']),
                   article=row['article'])

print(f'NetworkX graph: {kg_nx.number_of_nodes()} nodes, {kg_nx.number_of_edges()} edges')

In [ ]:
def push_to_neo4j(driver, triplets_df: pd.DataFrame, wikidata: dict):
    """Wipe the database and ingest all triplets with proper labels and properties."""
    rows = []
    for _, r in triplets_df.iterrows():
        h_info = wikidata.get(r['head']) or {}
        t_info = wikidata.get(r['tail']) or {}
        rows.append({
            'head': r['head'],
            'tail': r['tail'],
            'rel': relation_to_label(r['type']),
            'rel_raw': r['type'],
            'article': r['article'],
            'h_qid': h_info.get('qid'),
            'h_desc': h_info.get('description', ''),
            't_qid': t_info.get('qid'),
            't_desc': t_info.get('description', ''),
        })

    with driver.session() as session:
        session.run('MATCH (n) DETACH DELETE n')
        session.run('CREATE CONSTRAINT entity_name IF NOT EXISTS '
                    'FOR (e:Entity) REQUIRE e.name IS UNIQUE')
        session.run(
            '''
            UNWIND $rows AS row
            MERGE (h:Entity {name: row.head})
              ON CREATE SET h.qid = row.h_qid, h.description = row.h_desc
            MERGE (t:Entity {name: row.tail})
              ON CREATE SET t.qid = row.t_qid, t.description = row.t_desc
            CREATE (h)-[r:REL {type: row.rel, raw: row.rel_raw, article: row.article}]->(t)
            ''',
            rows=rows,
        )
    return len(rows)

if USE_NEO4J:
    n_written = push_to_neo4j(neo4j_driver, triplets_df, wikidata_cache)
    print(f'Wrote {n_written:,} triplets into Neo4j.')
else:
    print('Skipping Neo4j ingestion (no live database); using NetworkX backend.')

# 7. Query and Visualize the Knowledge Graph

Let's exercise the graph. Each demo shows the **Cypher** query you would run in Neo4j *and* an equivalent operation on the NetworkX fallback so the notebook works in both modes.

In [ ]:
def run_cypher(query: str, **params) -> pd.DataFrame:
    """Execute a Cypher query on the Neo4j driver and return a DataFrame."""
    if not USE_NEO4J:
        raise RuntimeError('Neo4j not connected.')
    with neo4j_driver.session() as session:
        return pd.DataFrame([r.data() for r in session.run(query, **params)])

### 7.1 Top entities by degree ("who is mentioned most?")

**Cypher**
```cypher
MATCH (e:Entity)
OPTIONAL MATCH (e)-[r]-()
RETURN e.name AS entity, e.qid AS qid, count(r) AS degree
ORDER BY degree DESC
LIMIT 10;
```

In [ ]:
if USE_NEO4J:
    top_entities_df = run_cypher('''
        MATCH (e:Entity)
        OPTIONAL MATCH (e)-[r]-()
        RETURN e.name AS entity, e.qid AS qid, count(r) AS degree
        ORDER BY degree DESC
        LIMIT 10
    ''')
else:
    deg = sorted(((n, kg_nx.degree(n)) for n in kg_nx.nodes()), key=lambda x: -x[1])[:10]
    top_entities_df = pd.DataFrame([
        {'entity': n, 'qid': kg_nx.nodes[n].get('qid'), 'degree': d} for n, d in deg
    ])
top_entities_df

### 7.2 Most common relation types in the graph

**Cypher**
```cypher
MATCH ()-[r:REL]->()
RETURN r.type AS relation, count(*) AS n
ORDER BY n DESC LIMIT 10;
```

In [ ]:
if USE_NEO4J:
    rel_df = run_cypher('''
        MATCH ()-[r:REL]->()
        RETURN r.type AS relation, count(*) AS n
        ORDER BY n DESC LIMIT 10
    ''')
else:
    rels = Counter(d['rel_label'] for _, _, d in kg_nx.edges(data=True))
    rel_df = pd.DataFrame(rels.most_common(10), columns=['relation', 'n'])
rel_df

### 7.3 Direct neighbors of a focal entity

Pick the most connected entity above and look at what it is linked to.

In [ ]:
focal_entity = top_entities_df.iloc[0]['entity']
print(f'Neighborhood of: {focal_entity}\n')

if USE_NEO4J:
    neighbors_df = run_cypher('''
        MATCH (e:Entity {name: $name})-[r:REL]-(n:Entity)
        RETURN startNode(r).name AS source, r.type AS relation,
               endNode(r).name   AS target, r.article AS article
        LIMIT 25
    ''', name=focal_entity)
else:
    rows = []
    for u, v, d in kg_nx.out_edges(focal_entity, data=True):
        rows.append({'source': u, 'relation': d['rel_label'], 'target': v, 'article': d.get('article')})
    for u, v, d in kg_nx.in_edges(focal_entity, data=True):
        rows.append({'source': u, 'relation': d['rel_label'], 'target': v, 'article': d.get('article')})
    neighbors_df = pd.DataFrame(rows).head(25)
neighbors_df

### 7.4 Shortest path between two entities

**Cypher**
```cypher
MATCH (a:Entity {name: $a}), (b:Entity {name: $b}),
      p = shortestPath((a)-[*..6]-(b))
RETURN [n IN nodes(p) | n.name] AS path;
```

In [ ]:
if len(top_entities_df) >= 2:
    a, b = top_entities_df.iloc[0]['entity'], top_entities_df.iloc[1]['entity']
    print(f'Path between {a!r} and {b!r}')
    if USE_NEO4J:
        path_df = run_cypher('''
            MATCH (a:Entity {name: $a}), (b:Entity {name: $b}),
                  p = shortestPath((a)-[*..6]-(b))
            RETURN [n IN nodes(p) | n.name] AS path
        ''', a=a, b=b)
        print(path_df.to_string(index=False) if len(path_df) else 'no path found')
    else:
        undirected = kg_nx.to_undirected()
        try:
            path = nx.shortest_path(undirected, a, b)
            print(' -> '.join(path))
        except nx.NetworkXNoPath:
            print('no path found')

### 7.5 Interactive subgraph visualization

We use `pyvis` to render an interactive HTML view of the focal entity's neighborhood. It works the same way whether the data is pulled from Neo4j or from NetworkX.

In [ ]:
def collect_subgraph(center: str, depth: int = 1) -> tuple[set[str], list[tuple]]:
    if USE_NEO4J:
        edges_df = run_cypher('''
            MATCH (e:Entity {name: $name})-[r:REL*1..%d]-(n:Entity)
            UNWIND r AS rel
            RETURN startNode(rel).name AS src, rel.type AS relation,
                   endNode(rel).name   AS dst
        ''' % depth, name=center)
        nodes = set(edges_df['src']).union(edges_df['dst'])
        edges = list(edges_df.itertuples(index=False, name=None))
    else:
        nodes = {center}
        edges = []
        frontier = {center}
        for _ in range(depth):
            new_frontier = set()
            for n in list(frontier):
                for u, v, d in kg_nx.out_edges(n, data=True):
                    edges.append((u, d['rel_label'], v))
                    new_frontier.add(v)
                for u, v, d in kg_nx.in_edges(n, data=True):
                    edges.append((u, d['rel_label'], v))
                    new_frontier.add(u)
            nodes.update(new_frontier)
            frontier = new_frontier
    return nodes, edges

nodes, edges = collect_subgraph(focal_entity, depth=1)
print(f'Subgraph centered on {focal_entity}: {len(nodes)} nodes / {len(edges)} edges')

net = Network(height='600px', width='100%', notebook=True,
              cdn_resources='in_line', directed=True, bgcolor='#ffffff')
net.force_atlas_2based()
for n in nodes:
    info = wikidata_cache.get(n) or {}
    color = colors[0] if n == focal_entity else colors[1]
    title = f"<b>{n}</b><br>QID: {info.get('qid', '-')}<br>{info.get('description', '')}"
    net.add_node(n, label=n[:40], title=title, color=color)
for src, rel, dst in edges:
    net.add_edge(src, dst, label=rel, arrows='to')

viz_path = CACHE_DIR / 'subgraph.html'
net.write_html(str(viz_path), open_browser=False, notebook=True)
IFrame(str(viz_path), width='100%', height='620px')

# 8. From Knowledge Graph to Chatbot – a basic agentic loop

A knowledge graph becomes really useful when we expose it to a language model. We are going to wire our graph into a *basic custom agentic loop* powered by [Anthropic's Claude](https://docs.anthropic.com).

The pattern is simple but surprisingly powerful:

```
user query
   |
   v
+----------+      tool_use     +-------------------+
|  Claude  |  ------------->   | KG tool functions |
| (LLM)    |  <-------------   | (search, paths,   |
+----------+    tool_result    |  neighbours...)   |
   |                           +-------------------+
   v
final answer
```

We define four tools that wrap our graph (`search_entities`, `get_neighbors`, `find_path`, `top_entities`). Claude decides which tools to call, looks at the results, and keeps iterating until it has enough information to answer. We cache every agent run to `cache/agent_runs.json` so re-executing the notebook is free.

If `ANTHROPIC_API_KEY` is not set in the environment the live demo cells are skipped gracefully – the notebook still runs end-to-end.

In [ ]:
ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL = os.environ.get('ANTHROPIC_MODEL', 'claude-sonnet-4-6')

if ANTHROPIC_API_KEY:
    anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
    HAS_ANTHROPIC = True
    print(f'Anthropic client ready (model={ANTHROPIC_MODEL}).')
else:
    anthropic_client = None
    HAS_ANTHROPIC = False
    print('ANTHROPIC_API_KEY is not set - chatbot demo cells will be skipped.')
    print('Export ANTHROPIC_API_KEY=... and re-run the chatbot cells to see it in action.')

## 8.1 Wrap the graph in Python tools

The agent does not talk to Neo4j or NetworkX directly: it calls regular Python functions that hide the backend. This means the *same* tool implementations work whether you are running against a live Neo4j or the in-memory NetworkX fallback we built in section 6.

In [ ]:
def _entity_exists(name: str) -> bool:
    if USE_NEO4J:
        with neo4j_driver.session() as s:
            r = s.run('MATCH (e:Entity {name: $name}) RETURN 1 LIMIT 1', name=name)
            return next(r, None) is not None
    return kg_nx.has_node(name)

def tool_search_entities(query: str, limit: int = 10) -> list[dict]:
    """Case-insensitive substring search of entity names."""
    q = query.lower()
    if USE_NEO4J:
        with neo4j_driver.session() as s:
            recs = s.run(
                'MATCH (e:Entity) WHERE toLower(e.name) CONTAINS $q '
                'RETURN e.name AS name, e.qid AS qid, e.description AS description LIMIT $limit',
                q=q, limit=limit,
            )
            return [r.data() for r in recs]
    out = []
    for n in kg_nx.nodes():
        if q in n.lower():
            out.append({
                'name': n,
                'qid': kg_nx.nodes[n].get('qid'),
                'description': kg_nx.nodes[n].get('description'),
            })
            if len(out) >= limit:
                break
    return out

def tool_get_neighbors(entity: str, limit: int = 20) -> dict:
    """Return all relations involving the entity (incoming and outgoing)."""
    if not _entity_exists(entity):
        return {'error': f'Entity {entity!r} not in graph. Try search_entities first.'}
    if USE_NEO4J:
        with neo4j_driver.session() as s:
            recs = s.run(
                'MATCH (e:Entity {name: $name})-[r:REL]-(o:Entity) '
                'RETURN startNode(r).name AS source, r.type AS relation, '
                '       endNode(r).name AS target, r.article AS article LIMIT $limit',
                name=entity, limit=limit,
            )
            edges = [r.data() for r in recs]
    else:
        edges = []
        for u, v, d in kg_nx.out_edges(entity, data=True):
            edges.append({'source': u, 'relation': d['rel_label'], 'target': v, 'article': d.get('article')})
        for u, v, d in kg_nx.in_edges(entity, data=True):
            edges.append({'source': u, 'relation': d['rel_label'], 'target': v, 'article': d.get('article')})
        edges = edges[:limit]
    info = wikidata_cache.get(entity) or {}
    return {
        'entity': entity,
        'qid': info.get('qid'),
        'description': info.get('description', ''),
        'edges': edges,
    }

def tool_find_path(source: str, target: str, max_hops: int = 4) -> dict:
    """Find a short path between two entities, if any."""
    for ent in (source, target):
        if not _entity_exists(ent):
            return {'error': f'Entity {ent!r} not in graph.'}
    if USE_NEO4J:
        with neo4j_driver.session() as s:
            recs = s.run(
                'MATCH (a:Entity {name: $a}), (b:Entity {name: $b}), '
                f'p = shortestPath((a)-[*..{int(max_hops)}]-(b)) '
                'RETURN [n IN nodes(p) | n.name] AS nodes, '
                '       [r IN relationships(p) | r.type] AS rels',
                a=source, b=target,
            )
            row = next(recs, None)
            if row is None:
                return {'nodes': None, 'message': 'no path found'}
            return {'nodes': row['nodes'], 'relations': row['rels']}
    try:
        undirected = kg_nx.to_undirected()
        nodes = nx.shortest_path(undirected, source, target)
        if len(nodes) - 1 > max_hops:
            return {'nodes': None, 'message': 'path too long'}
        rels = []
        for u, v in zip(nodes, nodes[1:]):
            if kg_nx.has_edge(u, v):
                rels.append(next(iter(kg_nx[u][v].values()))['rel_label'])
            elif kg_nx.has_edge(v, u):
                rels.append(next(iter(kg_nx[v][u].values()))['rel_label'])
            else:
                rels.append('?')
        return {'nodes': nodes, 'relations': rels}
    except nx.NetworkXNoPath:
        return {'nodes': None, 'message': 'no path found'}

def tool_top_entities(limit: int = 10) -> list[dict]:
    """Return the most connected entities in the graph."""
    if USE_NEO4J:
        with neo4j_driver.session() as s:
            recs = s.run(
                'MATCH (e:Entity) OPTIONAL MATCH (e)-[r]-() '
                'RETURN e.name AS name, e.qid AS qid, count(r) AS degree '
                'ORDER BY degree DESC LIMIT $limit',
                limit=limit,
            )
            return [r.data() for r in recs]
    deg = sorted(((n, kg_nx.degree(n)) for n in kg_nx.nodes()), key=lambda x: -x[1])[:limit]
    return [{'name': n, 'qid': kg_nx.nodes[n].get('qid'), 'degree': d} for n, d in deg]

KG_TOOL_FNS = {
    'search_entities': tool_search_entities,
    'get_neighbors':  tool_get_neighbors,
    'find_path':      tool_find_path,
    'top_entities':   tool_top_entities,
}

# Quick smoke test
print('search_entities("Valkyria") ->', tool_search_entities('Valkyria', limit=3))

## 8.2 Tool schemas

For Claude to know how to call our tools we must describe them with [JSON Schema](https://json-schema.org). The `description` field is what the model reads to decide *when* to use the tool, so make it count.

In [ ]:
KG_TOOL_SCHEMAS = [
    {
        'name': 'search_entities',
        'description': (
            'Case-insensitive substring search over entity names in the knowledge graph. '
            'Always call this FIRST when the user mentions an entity by name, so you can '
            'discover its exact form (which may include titles, full names, suffixes, etc.).'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string', 'description': 'Substring to match in entity names.'},
                'limit': {'type': 'integer', 'description': 'Max matches (default 10).'},
            },
            'required': ['query'],
        },
    },
    {
        'name': 'get_neighbors',
        'description': (
            'List all relations (incoming and outgoing) of one specific entity. '
            'The `entity` argument must be an EXACT match returned by search_entities.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'entity': {'type': 'string', 'description': 'Exact entity name.'},
                'limit':  {'type': 'integer', 'description': 'Max relations (default 20).'},
            },
            'required': ['entity'],
        },
    },
    {
        'name': 'find_path',
        'description': (
            'Find the shortest path (treating the graph as undirected) between two entities. '
            'Use this to discover whether and how two things are connected.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'source':   {'type': 'string', 'description': 'Source entity name.'},
                'target':   {'type': 'string', 'description': 'Target entity name.'},
                'max_hops': {'type': 'integer', 'description': 'Max path length (default 4).'},
            },
            'required': ['source', 'target'],
        },
    },
    {
        'name': 'top_entities',
        'description': 'List the most connected entities in the graph - useful as a starting point for open-ended exploration.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'limit': {'type': 'integer', 'description': 'Max results (default 10).'},
            },
        },
    },
]

## 8.3 The agentic loop

A *custom* agentic loop is just a `while` loop:

1. send the chat history (plus tool definitions) to Claude,
2. if the model returns one or more `tool_use` blocks, **execute the tools locally** and feed the results back as a `user` message with `tool_result` blocks,
3. repeat until the model returns `stop_reason == "end_turn"` (or we hit a safety budget on the number of iterations).

The Anthropic SDK ships a `tool_runner` helper that wraps exactly this loop, but writing it by hand makes the mechanics explicit and lets us trace, cache and customize every step.

In [ ]:
SYSTEM_PROMPT = (
    'You are a research assistant that answers questions about a knowledge graph extracted '
    'from a handful of Wikipedia articles via REBEL and Wikidata. '
    'The graph stores entities (people, places, dates, organisations...) linked by typed '
    "relations such as 'place of birth', 'member of sports team', 'cast member', etc. "
    'You have four tools: search_entities, get_neighbors, find_path, top_entities. '
    'Always call search_entities first to discover the EXACT entity name before using the others. '
    'When you have enough information, write a concise, factual answer that cites the entities and '
    'relations you used. If the graph does not contain the answer, say so explicitly - do not invent facts.'
)

AGENT_CACHE = CACHE_DIR / 'agent_runs.json'
if AGENT_CACHE.exists():
    with open(AGENT_CACHE) as f:
        agent_cache = json.load(f)
else:
    agent_cache = {}

def _serialize_content(blocks):
    out = []
    for b in blocks:
        if hasattr(b, 'model_dump'):
            out.append(b.model_dump())
        else:
            out.append(b)
    return out

def run_kg_agent(user_query: str, *, max_iterations: int = 8,
                 use_cache: bool = True, verbose: bool = True):
    """A basic custom agentic loop that lets Claude use KG tools to answer a question."""
    cache_key = f'{ANTHROPIC_MODEL}::{user_query}'
    if use_cache and cache_key in agent_cache:
        if verbose:
            print(f'[cache hit] {user_query!r}')
        cached = agent_cache[cache_key]
        return cached['answer'], cached['trace']

    if not HAS_ANTHROPIC:
        return '(ANTHROPIC_API_KEY missing - skipping live demo.)', []

    messages = [{'role': 'user', 'content': user_query}]
    trace = []
    answer = ''

    for step in range(max_iterations):
        response = anthropic_client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            tools=KG_TOOL_SCHEMAS,
            messages=messages,
        )
        trace.append({
            'step': step,
            'stop_reason': response.stop_reason,
            'content': _serialize_content(response.content),
        })
        messages.append({'role': 'assistant', 'content': response.content})

        if response.stop_reason == 'end_turn':
            answer = ''.join(getattr(b, 'text', '') for b in response.content
                             if getattr(b, 'type', '') == 'text')
            break

        if response.stop_reason != 'tool_use':
            answer = ''.join(getattr(b, 'text', '') for b in response.content)
            break

        tool_results = []
        for block in response.content:
            if getattr(block, 'type', '') == 'tool_use':
                if verbose:
                    print(f'[step {step}] -> {block.name}({block.input})')
                try:
                    result = KG_TOOL_FNS[block.name](**block.input)
                    payload = json.dumps(result, default=str, ensure_ascii=False)
                except Exception as exc:
                    payload = json.dumps({'error': str(exc)})
                tool_results.append({
                    'type': 'tool_result',
                    'tool_use_id': block.id,
                    'content': payload,
                })
        messages.append({'role': 'user', 'content': tool_results})
    else:
        answer = '(max iterations reached without final answer)'

    agent_cache[cache_key] = {'answer': answer, 'trace': trace}
    with open(AGENT_CACHE, 'w') as f:
        json.dump(agent_cache, f, indent=2, default=str)
    return answer, trace


def show_trace(trace):
    """Pretty-print the tool-use trace produced by run_kg_agent."""
    for entry in trace:
        print(f"-- step {entry['step']}  (stop_reason={entry['stop_reason']}) --")
        for block in entry['content']:
            if block.get('type') == 'tool_use':
                print(f"   TOOL  {block['name']}({block.get('input', {})})")
            elif block.get('type') == 'text':
                txt = (block.get('text') or '').strip().replace('\n', ' ')
                if txt:
                    print(f"   TEXT  {txt[:240]}")

## 8.4 Talk to the graph

Let's try three questions that exercise different tools.

In [ ]:
query = 'Tell me everything the knowledge graph knows about Charles Eaton.'
answer, trace = run_kg_agent(query)
print('\nANSWER:\n', answer)
print('\nTRACE:')
show_trace(trace)

In [ ]:
query = 'What can you tell me about the video game Valkyria Chronicles III based on the graph?'
answer, trace = run_kg_agent(query)
print('\nANSWER:\n', answer)
print('\nTRACE:')
show_trace(trace)

In [ ]:
query = 'Pick the two most connected entities in the graph and tell me whether they are related to each other.'
answer, trace = run_kg_agent(query)
print('\nANSWER:\n', answer)
print('\nTRACE:')
show_trace(trace)

## 8.5 Going further

A few directions to extend the chatbot:

* **More tools** – add `entity_description` (Wikidata blurb), `neighbors_by_relation(entity, relation)` for focused queries, or `cypher_query` for power users.
* **Streaming** – use `anthropic_client.messages.stream(...)` to display partial responses in real time.
* **Memory** – persist the `messages` list per session to enable multi-turn conversations.
* **Guardrails** – validate tool arguments, cap the result size before feeding it back to the model, and log all calls for evaluation.
* **GraphRAG** – combine these symbolic tools with vector retrieval over node embeddings (e.g. `Node2Vec`, `Sentence-BERT` over `entity.description`) for the best of both worlds.

### 7.6 Cleanup

Always close the Neo4j driver when done.

In [ ]:
if neo4j_driver is not None:
    neo4j_driver.close()
    print('Neo4j driver closed.')
else:
    print('NetworkX graph retained in memory as `kg_nx` for further exploration.')

## Recap

* **fastcoref** rewrote each article to resolve pronouns.
* **REBEL** distilled the resolved text into `(head, relation, tail)` triplets.
* **Wikidata** gave each entity a stable QID and a description.
* **Neo4j** (or its NetworkX fallback) stored the triplets and let us query them with Cypher.
* **pyvis** turned the result into an interactive picture.
* **Claude + a custom agentic loop** turned the graph into a chatbot that can search, traverse and reason about it through tool calls.

From here you can scale `NUM_ARTICLES` up, add additional relation extractors, ingest entity types from Wikidata to give your graph a richer schema, or layer graph algorithms (community detection, embeddings, GraphRAG) on top of what we built.

<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>